# Planning and task decomposition for AI agents

## Scenario: Adaptive RAG research agent

A product lead asks: **Research adaptive RAG and produce a technical report.** This notebook makes the planning process observable: a goal contract becomes a DAG, independent evidence tasks run first, a checkpoint assesses coverage, and a failed source creates a bounded graph patch.

**Learning outcomes:** decompose goals; schedule DAGs; separate planner and executor; use hierarchical and dynamic plans; add constraints, milestones, recovery, and terminal conditions. The deterministic implementation lives next to this notebook in [`lab.py`](lab.py).

## The planning system

![Dynamic planning graph](assets/planning-task-decomposition.svg)

A planner is allowed to propose typed work. It is *not* allowed to authorize tools, bypass a dependency, or decide that a failed source is good enough. Those decisions belong to validators, the scheduler, policy, and an explicit quality gate.

The diagram is a repository SVG rather than Mermaid so it renders consistently in GitHub notebooks and has an accessible description in its SVG metadata.

## 1. Write a goal contract before a plan

A request is not a plan. The contract fixes an audience, deliverable, required sections, evidence rules, allowed tools, cost/time/replan budgets, and terminal states. This keeps a long-horizon planner from optimizing for an attractive but untestable task list.

For this scenario, the report must explain what adaptive RAG changes, compare routing strategies, and discuss evaluation/trade-offs. It must cite evidence and must stop after the report is supported or safely escalated.

In [ ]:
from lab import Constraints, Task, initial_plan, make_state, topological_layers, validate_plan

state = make_state()
print(state.goal)
print(state.constraints)
print('\nTopological layers (each layer is eligible for parallel execution):')
for number, layer in enumerate(topological_layers(state.tasks.values()), start=1):
    print(f'  layer {number}: {layer}')

## 2. Goal decomposition and hierarchical planning

The top level owns stable intent: deliver an evidence-backed technical report. Under it, an evidence workstream collects the primary paper, RAG foundation, and implementation guidance. A comparison workstream consumes those artifacts; a quality milestone decides whether synthesis may run.

This is **hierarchical planning**: higher levels state *why* and *what*; leaves state the smallest verifiable action. Do not split tasks merely because a model can list steps. Split at artifact boundaries that another task can use without repeating the work.

In [ ]:
for task in initial_plan():
    print(f'{task.id:24} | kind={task.kind:10} | depends_on={task.depends_on or ("—",)}')
    print(f'  objective: {task.objective}')

## 3. DAG constraints are part of correctness

A directed acyclic graph expresses dependency management. Source reads are independent; comparison must wait for evidence; the report must wait for the checkpoint. A valid graph is not necessarily a good plan, but an invalid graph must never reach an executor.

The validator checks unique IDs, missing dependencies, task and retry budgets, allowed tool scopes, and cycles. The following deliberate cycle is rejected before any work is dispatched.

In [ ]:
bad = [
    Task('collect', 'Collect evidence', 'Read a source', depends_on=('report',)),
    Task('report', 'Write report', 'Synthesize the evidence', depends_on=('collect',), kind='synthesize', tool_scope=('synthesize',)),
]
print(validate_plan(bad, Constraints()))

## 4. Plan-and-execute is a control boundary

**Planner:** proposes `Task` records with objectives, dependencies, scopes, and attempt limits.

**Executor:** runs only ready, validated tasks and records a result or a typed failure.

**Evaluator/checkpoint:** returns a structured evidence gap, not a free-form feeling.

**Replanner:** makes the smallest graph mutation supported by that observation, revalidates it, and consumes a replan budget.

This is more reliable than asking a model to ‘think harder’ because failure recovery is externally inspectable and bounded. It also maps well to a `StateGraph`: planner node → validator → scheduler/executor → checkpoint → conditional edge to a replan node or synthesis. LangGraph documents the difference between code-owned workflows and dynamic agents, plus routing, parallelization, persistence, and evaluator-optimizer patterns.

In [ ]:
from lab import run_research_agent

healthy_run = run_research_agent(simulate_missing_source=False)
print('Events:')
print('\n'.join(healthy_run.events))
print('\nReport excerpt:')
print(healthy_run.findings['report'][:650])

## 5. Dynamic planning and bounded replanning

Now the implementation guidance source fails. A dangerous agent might loop on retry or continue with an unsupported conclusion. The safe run records the failure, adds a specific replacement source task, rewires only the comparison dependency, revalidates the DAG, and reruns downstream work.

A production replan trigger should be explicit: unavailable source, missing required evidence, a material conflict, changed user constraint, or a failed quality gate. Give each run maximum task count, per-task attempts, replan count, wall-clock time, cost, and a human escalation terminal state.

In [ ]:
replanned_run = run_research_agent(simulate_missing_source=True)
print('Replans:', replanned_run.replan_count)
print('\n'.join(replanned_run.events))
print('\nFinal task states:')
for task_id, status in replanned_run.status.items():
    print(f'  {task_id:24} {status.value}')

## 6. Long-horizon planning: milestones, durability, and recovery

For a report that lasts minutes or days, persist the goal contract, plan version, task statuses, task output hashes, source provenance, retry count, cost/time budget, and reason for every replan. Use idempotency keys around side effects. Resume from a checkpoint rather than replaying a finished source read or reissuing a ticket.

A useful milestone is not just a status label. It asks a testable question: *Are all required sections supported by attributable evidence?* If the answer is no, the system either performs an authorized, bounded patch or escalates.

**Do not use dynamic planning** when the path is known and stable. A deterministic workflow is usually cheaper, easier to test, and easier to audit. The planning premium must be justified by genuine runtime ambiguity.

## Production readiness checklist

- Validate schema, DAG, dependency IDs, tool scopes, retry limits, and budgets before dispatch.
- Treat retrieved text as data. It cannot create tasks, override policy, or authorize a tool.
- Evaluate the result and trajectory: section coverage, provenance, dependency correctness, redundant work, retry rate, latency, cost, and escalation quality.
- Store task-level traces and plan versions so a reviewer can answer: *Why did this task run? Why did the plan change? Which evidence supports this claim?*
- Keep human approval outside the planner for consequential actions.

## Exercises

1. Add a `security implications` report section and change the checkpoint so it detects missing coverage.
2. Add a maximum parallelism constraint; which tasks should the scheduler defer?
3. Simulate conflicting claims and design a `reconcile_conflict` task that preserves uncertainty rather than averaging claims.
4. Sketch a LangGraph implementation with a persisted state schema and a conditional `checkpoint → replan | synthesize | escalate` edge.
5. Compare this dynamic plan to a fixed three-step workflow. Which task characteristics justify the planning overhead?

## Sources

- [ReAct](https://arxiv.org/abs/2210.03629) — interleaves reasoning and external actions so observations can update a plan.
- [Plan-and-Solve Prompting](https://arxiv.org/abs/2305.04091) — proposes planning before execution to address missing-step errors.
- [Tree of Thoughts](https://arxiv.org/abs/2305.10601) — a search-oriented approach over intermediate states; apply only when search cost is justified.
- [Adaptive-RAG](https://arxiv.org/abs/2403.14403) — routes retrieval strategy according to question complexity.
- [LangGraph workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents) and [persistence](https://docs.langchain.com/oss/python/langgraph/persistence) — official patterns for routing, parallelization, evaluation loops, and durable state.